In [ ]:
!pip install peewee==3.14.8 arrow==1.2.3 click==8.0.4 numpy==1.26.4 pandas==2.2.3
!pip install git+https://github.com/jesse-ai/jesse.git

In [6]:
import os
import jesse.helpers as jh
from jesse.models import Candle
from peewee import SqliteDatabase

# Create required project files
if not os.path.exists('.env'):
    with open('.env', 'w') as f:
        f.write('SESSION_ID=colab_session\n')

if not os.path.exists('config.py'):
    with open('config.py', 'w') as f:
        f.write('config = {}\n')

os.makedirs('storage/candles', exist_ok=True)
os.makedirs('storage/database', exist_ok=True)

# Initialize and bind a local database to satisfy peewee/Jesse
db = SqliteDatabase('storage/database/database.sqlite')
db.connect()
Candle._meta.database = db
db.create_tables([Candle])

# Manually override Jesse's project check to allow running in Colab
def mock_is_jesse_project():
    return True

jh.is_jesse_project = mock_is_jesse_project

print("Jesse project environment initialized, database bound, and project check bypassed.")

Jesse project environment initialized, database bound, and project check bypassed.


Now that the environment is set up, you can re-run the cell starting with `research.get_candles`. If you still encounter directory errors, we may need to initialize a full Jesse project folder.

In [1]:
from jesse import utils, research
import jesse.helpers as jh
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

In [26]:
import jesse.helpers as jh
from jesse import research, utils
import os
import numpy as np
from datetime import datetime

# Ensure environment variables and directory structure are set
os.environ['SESSION_ID'] = 'colab_session'
def mock_is_jesse_project():
    return True
jh.is_jesse_project = mock_is_jesse_project

exchange = 'Binance Perpetual Futures'
start_date_str = '2024-10-01'
finish_date_str = '2024-10-02'
start_date = jh.date_to_timestamp(start_date_str)
finish_date = jh.date_to_timestamp(finish_date_str)

# Jesse requires 1m candles for database storage
def inject_mock_data(symbol):
    print(f"Injecting 1m mock data for {symbol}...")
    count = 1440
    base_ts = start_date
    candles = []
    np.random.seed(42 if 'ETH' in symbol else 24)
    price = 2500 if 'ETH' in symbol else 20
    for i in range(count):
        ts = base_ts + (i * 60 * 1000)
        price += np.random.randn() * (price * 0.0005)
        candles.append([ts, price, price, price, price, 100])

    research.store_candles(np.array(candles), exchange, symbol)

try:
    inject_mock_data('ETH-USDT')
    inject_mock_data('ETC-USDT')

    res1 = research.get_candles(exchange, 'ETH-USDT', '15m', start_date, finish_date)
    res2 = research.get_candles(exchange, 'ETC-USDT', '15m', start_date, finish_date)

    # Handle case where return might be (status, candles) or just candles
    global c1, c2, times
    c1 = res1[1] if isinstance(res1, tuple) else res1
    c2 = res2[1] if isinstance(res2, tuple) else res2

    if c1 is not None and c2 is not None and len(c1) > 0:
        min_len = min(len(c1), len(c2))
        c1 = c1[:min_len]
        c2 = c2[:min_len]

        times = [datetime.fromtimestamp(c[0] / 1000) for c in c1]
        print(f"Data Ready: {len(c1)} candles available. 'c1', 'c2', and 'times' are fixed.")
    else:
        print("Candle data extraction failed.")
except Exception as e:
    print(f"Error: {e}")

Injecting 1m mock data for ETH-USDT...
Injecting 1m mock data for ETC-USDT...
Data Ready: 96 candles available. 'c1', 'c2', and 'times' are fixed.


In [31]:
# Calculate returns and drop the first NaN element to match dimensions
eth_returns = utils.prices_to_returns(c1[:, 2])[1:]
etc_returns = utils.prices_to_returns(c2[:, 2])[1:]

print(f"Returns calculated. Shape: {eth_returns.shape}")

Returns calculated. Shape: (95,)


In [37]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))
# times[1:] matches the length of the returns (95 elements)
plt.plot(times[1:], eth_returns, color='blue', label='ETH Returns')
plt.plot(times[1:], etc_returns, color='red', label='ETC Returns')
plt.title('ETH vs ETC Returns (Mock Data)')
plt.legend()
plt.grid(True)
plt.show()

In [38]:
# Calculate spread and z-score from clean returns
spread = eth_returns - etc_returns
z_score = utils.z_score(spread)
mean_z = np.nanmean(z_score)

# Plot Z-Score
plt.figure(figsize=(15, 6))
plt.plot(times[1:], z_score, color='purple', label='z-score')
plt.axhline(mean_z, color='black', linestyle='--', label='mean')
plt.axhline(1.2, color='red', linestyle='--', label='Upper Threshold (1.2)')
plt.axhline(-1.2, color='green', linestyle='--', label='Lower Threshold (-1.2)')
plt.title('Z-Score of the Spread')
plt.legend()
plt.grid(True)
plt.show()

In [35]:
# Perform Cointegration test using clean returns
is_cointegrated = utils.are_cointegrated(eth_returns, etc_returns)
print(f"Are ETH and ETC cointegrated? {is_cointegrated}")

Are ETH and ETC cointegrated? True


### Визуализация торговых сигналов
Ниже представлен код, который помечает точки входа (Long/Short) и точки выхода (Exit) на основе рассчитанного Z-Score.

In [36]:
threshold = 1.2
entry_long = z_score < -threshold
entry_short = z_score > threshold
exit = np.abs(z_score) < 0.1

plt.figure(figsize=(15, 7))
plt.plot(times[1:], z_score, color='grey', alpha=0.5, label='Z-Score')

# Точки входа
plt.scatter(np.array(times[1:])[entry_long], z_score[entry_long], color='green', marker='^', label='Long ETH / Short ETC')
plt.scatter(np.array(times[1:])[entry_short], z_score[entry_short], color='red', marker='v', label='Short ETH / Long ETC')

plt.axhline(threshold, color='red', linestyle='--')
plt.axhline(-threshold, color='green', linestyle='--')
plt.axhline(0, color='black', lw=1)

plt.title('Торговые сигналы на основе Z-Score')
plt.legend()
plt.grid(True)
plt.show()

### Векторный бэктест стратегии
В этом блоке мы рассчитываем накопленную доходность, предполагая, что мы входим в позицию при достижении порога и выходим при возврате к среднему.

In [39]:
import pandas as pd

# Создаем DataFrame для удобства расчетов
backtest_df = pd.DataFrame({
    'z_score': z_score,
    'spread_return': spread / 100 # Приводим к десятичному виду, если spread в процентах
}, index=times[1:])

# Логика позиций
backtest_df['position'] = 0
backtest_df.loc[backtest_df['z_score'] < -threshold, 'position'] = 1  # Long ETH, Short ETC
backtest_df.loc[backtest_df['z_score'] > threshold, 'position'] = -1  # Short ETH, Long ETC

# Заполняем позиции (удерживаем до сигнала выхода 0)
# В упрощенном варианте просто используем текущие сигналы
backtest_df['strategy_returns'] = backtest_df['position'].shift(1) * backtest_df['spread_return']
backtest_df['cumulative_returns'] = (1 + backtest_df['strategy_returns'].fillna(0)).cumprod()

# Визуализация результатов
plt.figure(figsize=(15, 6))
plt.plot(backtest_df.index, backtest_df['cumulative_returns'], color='orange', label='Strategy Equity')
plt.title('Результаты бэктеста (Накопленная доходность)')
plt.xlabel('Дата')
plt.ylabel('Множитель капитала')
plt.legend()
plt.grid(True)
plt.show()

final_return = (backtest_df['cumulative_returns'].iloc[-1] - 1) * 100
print(f"Итоговая доходность за период: {final_return:.2f}%")

Итоговая доходность за период: 0.68%


### Что важно учитывать при торговле:
1. **Комиссии:** Поскольку это высокочастотная стратегия, комиссии биржи могут «съесть» всю прибыль. Нужно использовать лимитные ордера.
2. **Плечо (Leverage):** Так как спред движется на доли процентов, трейдеры часто используют плечо, чтобы увеличить доходность (но это повышает риски).
3. **Риск расхождения:** Cointegration может сломаться. Если спред продолжает расти и не возвращается к средней, позицию нужно закрывать по стоп-лоссу.